In [ ]:
!pip install konlpy
!pip install mecab-python
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)


In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from konlpy.tag import Mecab
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
df = pd.read_csv('samples.csv')
df.head()

In [ ]:
len(df)

In [ ]:
labels = []

for i in range(len(df)):
    label = df['output'][i].split(',')
    labels.append(label)

labels[:5], len(labels)

In [ ]:
labels = pd.DataFrame(labels)

In [ ]:
labels

In [ ]:
df = pd.concat([df['user_prompt'], labels], axis=1)
df

In [ ]:
print(df.isnull().sum())

In [ ]:
df['user_prompt'] = df['user_prompt'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","", regex=True)
df[:5]

In [ ]:
df[0] = df[0].replace(['사실형', '추론형', '대화형', '예측형'], [0,1,2,3])
df[1] = df[1].replace(['긍정', '부정', '미정'], [0,1,2])
df[2] = df[2].replace(['현재', '과거', '미래'], [0,1,2])
df[3] = df[3].replace(['확실', '불확실'], [0,1])

In [ ]:
df.head()

In [ ]:
df['user_prompt'] = df['user_prompt'].str.replace('^ +', "", regex=True)
df['user_prompt'].replace('', np.nan, inplace=True)
print(df.isnull().sum())

In [ ]:
X_data = df['user_prompt']
y_data_0 = df[0]
y_data_1 = df[1]
y_data_2 = df[2]
y_data_3 = df[3]

In [ ]:
label_0 = df[0].unique()
label_1 = df[1].unique()
label_2 = df[2].unique()
label_3 = df[3].unique()

label_0, label_1, label_2, label_3

In [ ]:
X_train_0, X_test_0, y_train_0, y_test_0 = train_test_split(X_data, y_data_0, test_size=0.5, random_state=0, stratify=y_data_0)
X_train_0, X_valid_0, y_train_0, y_valid_0 = train_test_split(X_train_0, y_train_0, test_size=.2, random_state=0, stratify=y_train_0)

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_data, y_data_1, test_size=0.5, random_state=0, stratify=y_data_1)
X_train_1, X_valid_1, y_train_1, y_valid_1 = train_test_split(X_train_1, y_train_1, test_size=.2, random_state=0, stratify=y_train_1)

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_data, y_data_2, test_size=0.5, random_state=0, stratify=y_data_2)
X_train_2, X_valid_2, y_train_2, y_valid_2 = train_test_split(X_train_2, y_train_2, test_size=.2, random_state=0, stratify=y_train_2)

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_data, y_data_3, test_size=0.5, random_state=0, stratify=y_data_3)
X_train_3, X_valid_3, y_train_3, y_valid_3 = train_test_split(X_train_3, y_train_3, test_size=.2, random_state=0, stratify=y_train_3)

print('--------유형 훈련 데이터의 비율-----------')
print(f'사실형 리뷰 = {round(y_train_0.value_counts()[0]/len(y_train_0) * 100,3)}%')
print(f'추론형 리뷰 = {round(y_train_0.value_counts()[1]/len(y_train_0) * 100,3)}%')
print(f'대화형 리뷰 = {round(y_train_0.value_counts()[2]/len(y_train_0) * 100,3)}%')
print(f'예측형 리뷰 = {round(y_train_0.value_counts()[3]/len(y_train_0) * 100,3)}%')
print('--------유형 검증 데이터의 비율-----------')
print(f'사실형 리뷰 = {round(y_valid_0.value_counts()[0]/len(y_valid_0) * 100,3)}%')
print(f'추론형 리뷰 = {round(y_valid_0.value_counts()[1]/len(y_valid_0) * 100,3)}%')
print(f'대화형 리뷰 = {round(y_valid_0.value_counts()[2]/len(y_valid_0) * 100,3)}%')
print(f'예측형 리뷰 = {round(y_valid_0.value_counts()[3]/len(y_valid_0) * 100,3)}%')
print('--------유형 테스트 데이터의 비율-----------')
print(f'사실형 리뷰 = {round(y_test_0.value_counts()[0]/len(y_test_0) * 100,3)}%')
print(f'추론형 리뷰 = {round(y_test_0.value_counts()[1]/len(y_test_0) * 100,3)}%')
print(f'대화형 리뷰 = {round(y_test_0.value_counts()[2]/len(y_test_0) * 100,3)}%')
print(f'예측형 리뷰 = {round(y_test_0.value_counts()[3]/len(y_test_0) * 100,3)}%')

print('--------극성 훈련 데이터의 비율-----------')
print(f'긍정 리뷰 = {round(y_train_1.value_counts()[0]/len(y_train_1) * 100,3)}%')
print(f'부정 리뷰 = {round(y_train_1.value_counts()[1]/len(y_train_1) * 100,3)}%')
print(f'미정 리뷰 = {round(y_train_1.value_counts()[2]/len(y_train_1) * 100,3)}%')
print('--------극성 검증 데이터의 비율-----------')
print(f'긍정 리뷰 = {round(y_valid_1.value_counts()[0]/len(y_valid_1) * 100,3)}%')
print(f'부정 리뷰 = {round(y_valid_1.value_counts()[1]/len(y_valid_1) * 100,3)}%')
print(f'미정 리뷰 = {round(y_valid_1.value_counts()[2]/len(y_valid_1) * 100,3)}%')
print('--------극성 테스트 데이터의 비율-----------')
print(f'긍정 리뷰 = {round(y_test_1.value_counts()[0]/len(y_test_1) * 100,3)}%')
print(f'부정 리뷰 = {round(y_test_1.value_counts()[1]/len(y_test_1) * 100,3)}%')
print(f'미정 리뷰 = {round(y_test_1.value_counts()[2]/len(y_test_1) * 100,3)}%')

print('--------시제 훈련 데이터의 비율-----------')
print(f'과거 리뷰 = {round(y_train_2.value_counts()[0]/len(y_train_2) * 100,3)}%')
print(f'현재 리뷰 = {round(y_train_2.value_counts()[1]/len(y_train_2) * 100,3)}%')
print(f'미래 리뷰 = {round(y_train_2.value_counts()[2]/len(y_train_2) * 100,3)}%')
print('--------시제 검증 데이터의 비율-----------')
print(f'과거 리뷰 = {round(y_valid_2.value_counts()[0]/len(y_valid_2) * 100,3)}%')
print(f'현재 리뷰 = {round(y_valid_2.value_counts()[1]/len(y_valid_2) * 100,3)}%')
print(f'미래 리뷰 = {round(y_valid_2.value_counts()[2]/len(y_valid_2) * 100,3)}%')
print('--------시제 테스트 데이터의 비율-----------')
print(f'과거 리뷰 = {round(y_test_2.value_counts()[0]/len(y_test_2) * 100,3)}%')
print(f'현재 리뷰 = {round(y_test_2.value_counts()[1]/len(y_test_2) * 100,3)}%')
print(f'미래 리뷰 = {round(y_test_2.value_counts()[2]/len(y_test_2) * 100,3)}%')

print('--------확실성 훈련 데이터의 비율-----------')
print(f"확실 리뷰 = {round(y_train_3.value_counts()[0]/len(y_train_3) * 100,3)}%'")
print(f'불확실 리뷰 = {round(y_train_3.value_counts()[1]/len(y_train_3) * 100,3)}%')
print('--------확실성 검증 데이터의 비율-----------')
print(f'확실 리뷰 = {round(y_valid_3.value_counts()[0]/len(y_valid_3) * 100,3)}%')
print(f'불확실 리뷰 = {round(y_valid_3.value_counts()[1]/len(y_valid_3) * 100,3)}%')
print('--------확실성 테스트 데이터의 비율-----------')
print(f'확실 리뷰 = {round(y_test_3.value_counts()[0]/len(y_test_3) * 100,3)}%')
print(f'불확실 리뷰 = {round(y_test_3.value_counts()[1]/len(y_test_3) * 100,3)}%')

In [ ]:
stopwords = ['도', '는', '다', '의', '가', '이', '은', '한', '에', '하', '고', '을', '를', '인', '듯', '과', '와', '네', '들', '듯', '지', '임', '게']

In [ ]:
mecab = Mecab()

In [ ]:
X_train_0

In [ ]:
X_train_0

In [ ]:
def tokenize(sentences):
  tokenized_sentences = []
  for sent in tqdm(sentences):
    tokenized_sent = mecab.morphs(sent)
    stopwords_removed_sentence = [word for word in tokenized_sent if not word in stopwords]
    tokenized_sentences.append(stopwords_removed_sentence)
  return tokenized_sentences

tokenized_X_train_0 = tokenize(X_train_0)
tokenized_X_valid_0 = tokenize(X_valid_0)
tokenized_X_test_0 = tokenize(X_test_0)
tokenized_X_train_1 = tokenize(X_train_1)
tokenized_X_valid_1 = tokenize(X_valid_1)
tokenized_X_test_1 = tokenize(X_test_1)
tokenized_X_train_2 = tokenize(X_train_2)
tokenized_X_valid_2 = tokenize(X_valid_2)
tokenized_X_test_2 = tokenize(X_test_2)
tokenized_X_train_3 = tokenize(X_train_3)
tokenized_X_valid_3 = tokenize(X_valid_3)
tokenized_X_test_3 = tokenize(X_test_3)

In [ ]:
for sent in tokenized_X_test_0[:2]:
    print(sent)

In [ ]:
X_test_0[:2]

In [ ]:
word_list = []
for sent in tokenized_X_train_0:
    for word in sent:
        word_list.append(word)

word_counts = Counter(word_list)
print('총 단어수 :', len(word_counts))

In [ ]:
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
print('등장 빈도수 상위 10개 단어')
print(vocab[:10])

In [ ]:
threshold = 3
total_cnt = len(word_counts) # 단어의 수
rare_cnt = 0 # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0 # 훈련 데이터의 전체 단어 빈도수 총 합
rare_freq = 0 # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수의 총 합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in word_counts.items():
    total_freq = total_freq + value

    # 단어의 등장 빈도수가 threshold보다 작으면
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold - 1, rare_cnt))
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)


In [ ]:
threshold = 2
total_cnt = len(word_counts) # 단어의 수
rare_cnt = 0 # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0 # 훈련 데이터의 전체 단어 빈도수 총 합
rare_freq = 0 # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수의 총 합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in word_counts.items():
    total_freq = total_freq + value

    # 단어의 등장 빈도수가 threshold보다 작으면
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold - 1, rare_cnt))
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)


In [ ]:
word_to_index = {}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

for index, word in enumerate(vocab):
    word_to_index[word] = index + 2

vocab_size = len(word_to_index)

vocab_size

In [ ]:
def texts_to_sequences(tokenized_X_data, word_to_index):
  encoded_X_data = []
  for sent in tokenized_X_data:
    index_sequences = []
    for word in sent:
      try:
          index_sequences.append(word_to_index[word])
      except KeyError:
          index_sequences.append(word_to_index['<UNK>'])
    encoded_X_data.append(index_sequences)
  return encoded_X_data

encoded_X_train_0 = texts_to_sequences(tokenized_X_train_0, word_to_index)
encoded_X_valid_0 = texts_to_sequences(tokenized_X_valid_0, word_to_index)
encoded_X_test_0 = texts_to_sequences(tokenized_X_test_0, word_to_index)

encoded_X_train_1 = texts_to_sequences(tokenized_X_train_1, word_to_index)
encoded_X_valid_1 = texts_to_sequences(tokenized_X_valid_1, word_to_index)
encoded_X_test_1 = texts_to_sequences(tokenized_X_test_1, word_to_index)

encoded_X_train_2 = texts_to_sequences(tokenized_X_train_2, word_to_index)
encoded_X_valid_2 = texts_to_sequences(tokenized_X_valid_2, word_to_index)
encoded_X_test_2 = texts_to_sequences(tokenized_X_test_2, word_to_index)

encoded_X_train_3 = texts_to_sequences(tokenized_X_train_3, word_to_index)
encoded_X_valid_3 = texts_to_sequences(tokenized_X_valid_3, word_to_index)
encoded_X_test_3 = texts_to_sequences(tokenized_X_test_3, word_to_index)

In [ ]:
encoded_X = encoded_X_test_0 + encoded_X_train_0 + encoded_X_valid_0

In [ ]:
print('문장의 최대 길이 :',max(len(review) for review in encoded_X))
print('문장의 평균 길이 :',sum(map(len, encoded_X))/len(encoded_X))
plt.hist([len(review) for review in encoded_X], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
def pad_sequences(sentences, max_len):
  features = np.zeros((len(sentences), max_len), dtype=int)
  for index, sentence in enumerate(sentences):
    if len(sentence) != 0:
      features[index, :len(sentence)] = np.array(sentence)[:max_len]
  return features

max_len = 167

padded_X_train_0 = pad_sequences(encoded_X_train_0, max_len=max_len)
padded_X_valid_0 = pad_sequences(encoded_X_valid_0, max_len=max_len)
padded_X_test_0 = pad_sequences(encoded_X_test_0, max_len=max_len)

print('훈련 데이터0의 크기 :', padded_X_train_0.shape)
print('검증 데이터0의 크기 :', padded_X_valid_0.shape)
print('테스트 데이터0의 크기 :', padded_X_test_0.shape)

padded_X_train_1 = pad_sequences(encoded_X_train_1, max_len=max_len)
padded_X_valid_1 = pad_sequences(encoded_X_valid_1, max_len=max_len)
padded_X_test_1 = pad_sequences(encoded_X_test_1, max_len=max_len)

print('훈련 데이터1의 크기 :', padded_X_train_1.shape)
print('검증 데이터1의 크기 :', padded_X_valid_1.shape)
print('테스트 데이터1의 크기 :', padded_X_test_1.shape)

padded_X_train_2 = pad_sequences(encoded_X_train_2, max_len=max_len)
padded_X_valid_2 = pad_sequences(encoded_X_valid_2, max_len=max_len)
padded_X_test_2 = pad_sequences(encoded_X_test_2, max_len=max_len)

print('훈련 데이터2의 크기 :', padded_X_train_2.shape)
print('검증 데이터2의 크기 :', padded_X_valid_2.shape)
print('테스트 데이터2의 크기 :', padded_X_test_2.shape)

padded_X_train_3 = pad_sequences(encoded_X_train_3, max_len=max_len)
padded_X_valid_3 = pad_sequences(encoded_X_valid_3, max_len=max_len)
padded_X_test_3 = pad_sequences(encoded_X_test_3, max_len=max_len)

print('훈련 데이터3의 크기 :', padded_X_train_3.shape)
print('검증 데이터3의 크기 :', padded_X_valid_3.shape)
print('테스트 데이터3의 크기 :', padded_X_test_3.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
print("cpu와 cuda 중 다음 기기로 학습함:", device)


In [ ]:
train_label_tensor_0 = torch.tensor(np.array(y_train_0))
valid_label_tensor_0 = torch.tensor(np.array(y_valid_0))
test_label_tensor_0 = torch.tensor(np.array(y_test_0))

train_label_tensor_1 = torch.tensor(np.array(y_train_1))
valid_label_tensor_1 = torch.tensor(np.array(y_valid_1))
test_label_tensor_1 = torch.tensor(np.array(y_test_1))

train_label_tensor_2 = torch.tensor(np.array(y_train_2))
valid_label_tensor_2 = torch.tensor(np.array(y_valid_2))
test_label_tensor_2 = torch.tensor(np.array(y_test_2))

train_label_tensor_3 = torch.tensor(np.array(y_train_3))
valid_label_tensor_3 = torch.tensor(np.array(y_valid_3))
test_label_tensor_3 = torch.tensor(np.array(y_test_3))

In [ ]:
print(train_label_tensor_0[-5:])

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim) # output_dim = 분류하고자하는 카테고리의 개수

    def forward(self, x):
        # x: (batch_size, seq_length) == (32, 500)
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim) == (32, 500, 100) == (데이터의 개수, 문장길이, 단어 벡터의 차원)
        gru_out, hidden = self.gru(embedded)  # gru_out: (batch_size, seq_length, hidden_dim), hidden: (1, batch_size, hidden_dim)
        last_hidden = hidden.squeeze(0)  # (batch_size, hidden_dim)
        logits = self.fc(last_hidden)  # (batch_size, output_dim)
        return logits

In [ ]:
encoded_train_0 = torch.tensor(padded_X_train_0).to(torch.int64)
train_dataset_0 = torch.utils.data.TensorDataset(encoded_train_0, train_label_tensor_0)
train_dataloader_0 = torch.utils.data.DataLoader(train_dataset_0, shuffle=True, batch_size=32)

encoded_test_0 = torch.tensor(padded_X_test_0).to(torch.int64)
test_dataset_0 = torch.utils.data.TensorDataset(encoded_test_0, test_label_tensor_0)
test_dataloader_0 = torch.utils.data.DataLoader(test_dataset_0, shuffle=True, batch_size=1)

encoded_valid_0 = torch.tensor(padded_X_valid_0).to(torch.int64)
valid_dataset_0 = torch.utils.data.TensorDataset(encoded_valid_0, valid_label_tensor_0)
valid_dataloader_0 = torch.utils.data.DataLoader(valid_dataset_0, shuffle=True, batch_size=1)

encoded_train_1 = torch.tensor(padded_X_train_1).to(torch.int64)
train_dataset_1 = torch.utils.data.TensorDataset(encoded_train_1, train_label_tensor_1)
train_dataloader_1 = torch.utils.data.DataLoader(train_dataset_1, shuffle=True, batch_size=32)

encoded_test_1 = torch.tensor(padded_X_test_1).to(torch.int64)
test_dataset_1 = torch.utils.data.TensorDataset(encoded_test_1, test_label_tensor_1)
test_dataloader_1 = torch.utils.data.DataLoader(test_dataset_1, shuffle=True, batch_size=1)

encoded_valid_1 = torch.tensor(padded_X_valid_1).to(torch.int64)
valid_dataset_1 = torch.utils.data.TensorDataset(encoded_valid_1, valid_label_tensor_1)
valid_dataloader_1 = torch.utils.data.DataLoader(valid_dataset_1, shuffle=True, batch_size=1)

encoded_train_2 = torch.tensor(padded_X_train_2).to(torch.int64)
train_dataset_2 = torch.utils.data.TensorDataset(encoded_train_2, train_label_tensor_2)
train_dataloader_2 = torch.utils.data.DataLoader(train_dataset_2, shuffle=True, batch_size=32)

encoded_test_2 = torch.tensor(padded_X_test_2).to(torch.int64)
test_dataset_2 = torch.utils.data.TensorDataset(encoded_test_2, test_label_tensor_2)
test_dataloader_2 = torch.utils.data.DataLoader(test_dataset_2, shuffle=True, batch_size=1)

encoded_valid_2 = torch.tensor(padded_X_valid_2).to(torch.int64)
valid_dataset_2 = torch.utils.data.TensorDataset(encoded_valid_2, valid_label_tensor_2)
valid_dataloader_2 = torch.utils.data.DataLoader(valid_dataset_2, shuffle=True, batch_size=1)

encoded_train_3 = torch.tensor(padded_X_train_3).to(torch.int64)
train_dataset_3 = torch.utils.data.TensorDataset(encoded_train_3, train_label_tensor_3)
train_dataloader_3 = torch.utils.data.DataLoader(train_dataset_3, shuffle=True, batch_size=32)

encoded_test_3 = torch.tensor(padded_X_test_3).to(torch.int64)
test_dataset_3 = torch.utils.data.TensorDataset(encoded_test_3, test_label_tensor_3)
test_dataloader_3 = torch.utils.data.DataLoader(test_dataset_3, shuffle=True, batch_size=1)

encoded_valid_3 = torch.tensor(padded_X_valid_3).to(torch.int64)
valid_dataset_3 = torch.utils.data.TensorDataset(encoded_valid_3, valid_label_tensor_3)
valid_dataloader_3 = torch.utils.data.DataLoader(valid_dataset_3, shuffle=True, batch_size=1)



In [ ]:
total_batch = len(train_dataloader_0)
total_batch

In [ ]:
embedding_dim = 100
hidden_dim = 128
output_dim_0 = 4
output_dim_1 = 3
output_dim_2 = 3
output_dim_3 = 2

model_0 = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim_0)
model_0.to(device)
model_1 = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim_1)
model_1.to(device)
model_2 = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim_2)
model_2.to(device)
model_3 = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim_3)
model_3.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer_0 = torch.optim.Adam(model_0.parameters(), lr=0.001)
optimizer_1 = torch.optim.Adam(model_1.parameters(), lr=0.001)
optimizer_2 = torch.optim.Adam(model_2.parameters(), lr=0.001)
optimizer_3 = torch.optim.Adam(model_3.parameters(), lr=0.001)

In [ ]:
def calculate_accuracy(logits, labels):
    # _, predicted = torch.max(logits, 1)
    predicted = torch.argmax(logits, dim=1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total
    return accuracy

In [ ]:
def evaluate(model, valid_dataloader, criterion, device):
    val_loss = 0
    val_correct = 0
    val_total = 0

    model.eval()
    with torch.no_grad():
        # 데이터로더로부터 배치 크기만큼의 데이터를 연속으로 로드
        for batch_X, batch_y in valid_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            # 모델의 예측값
            logits = model(batch_X)

            # 손실을 계산
            loss = criterion(logits, batch_y)

            # 정확도와 손실을 계산함
            val_loss += loss.item()
            val_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
            val_total += batch_y.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= len(valid_dataloader)

    return val_loss, val_accuracy


In [ ]:
num_epochs = 10  # 총 학습을 몇 번 반복할 것인지 설정하는 변수, 여기서는 5번 반복합니다.

# Training loop
def train_loop(model, train_dataloader, optimizer, valid_dataloader, model_name):
    # Training loop
    best_val_loss = float('inf')  # 검증 손실의 최저 값을 추적하기 위한 변수로, 초기값은 매우 큰 값으로 설정합니다.
    for epoch in range(num_epochs):  # 설정된 에포크 수만큼 반복합니다.
        # Training
        train_loss = 0  # 에포크 동안의 전체 훈련 손실을 저장할 변수입니다.
        train_correct = 0  # 에포크 동안 올바르게 예측된 샘플의 수를 저장할 변수입니다.
        train_total = 0  # 에포크 동안 처리된 총 샘플 수를 저장할 변수입니다.
        model.train()  # 모델을 훈련 모드로 설정합니다.

        for batch_X, batch_y in train_dataloader:  # 훈련 데이터셋을 배치 단위로 반복합니다.
            # Forward pass
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)  # 배치 데이터를 GPU와 같은 장치에 올립니다.
            # batch_X.shape == (batch_size, max_len)
            logits = model(batch_X)  # 모델에 입력 데이터를 넣어 예측값(logits)을 계산합니다.

            # Compute loss
            loss = criterion(logits, batch_y)  # 예측값과 실제 값 간의 손실(loss)을 계산합니다.

            # Backward pass and optimization
            optimizer.zero_grad()  # 이전 배치에서 계산된 기울기(gradient)를 초기화합니다.
            loss.backward()  # 역전파를 통해 기울기를 계산합니다.
            optimizer.step()  # 계산된 기울기를 사용하여 모델의 파라미터를 업데이트합니다.

            # Calculate training accuracy and loss
            train_loss += loss.item()  # 현재 배치의 손실을 전체 훈련 손실에 추가합니다.
            train_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)  # 정확도를 계산하여 올바르게 예측된 샘플 수를 추가합니다.
            train_total += batch_y.size(0)  # 현재 배치의 샘플 수를 전체 샘플 수에 추가합니다.

        train_accuracy = train_correct / train_total  # 전체 훈련 데이터에 대한 정확도를 계산합니다.
        train_loss /= len(train_dataloader)  # 배치 수로 나누어 평균 훈련 손실을 계산합니다.

        # Validation
        val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)  # 검증 데이터로 모델을 평가하여 손실과 정확도를 계산합니다.

        print(f'Epoch {epoch+1}/{num_epochs}:')  # 현재 에포크 번호와 총 에포크 수를 출력합니다.
        print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')  # 훈련 손실과 정확도를 출력합니다.
        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')  # 검증 손실과 정확도를 출력합니다.

        # 검증 손실이 최소일 때 체크포인트 저장
        if val_loss < best_val_loss:  # 현재 검증 손실이 이전의 최저 손실보다 낮으면
            print(f'Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. 체크포인트를 저장합니다.')  # 손실이 개선되었음을 출력합니다.
            best_val_loss = val_loss  # 최저 검증 손실을 현재 손실로 업데이트합니다.
            torch.save(model.state_dict(), model_name)  # 현재 모델의 가중치를 파일로 저장합니다.


In [ ]:
train_loop(model_0, train_dataloader_0, optimizer_0, valid_dataloader_0, 'best_model_0_checkpoint.pth')

In [ ]:
model_0.load_state_dict(torch.load('best_model_0_checkpoint.pth'))
model_0.to(device)

In [ ]:
# 검증 데이터에 대한 정확도와 손실 계산
val_loss_0, val_accuracy_0 = evaluate(model_0, valid_dataloader_0, criterion, device)

print(f'Best model validation loss: {val_loss_0:.4f}')
print(f'Best model validation accuracy: {val_accuracy_0:.4f}')


In [ ]:
train_loop(model_1, train_dataloader_1, optimizer_1, valid_dataloader_1, 'best_model_1_checkpoint.pth')

model_1.load_state_dict(torch.load('best_model_1_checkpoint.pth'))
model_1.to(device)

# 검증 데이터에 대한 정확도와 손실 계산
val_loss_1, val_accuracy_1 = evaluate(model_1, valid_dataloader_1, criterion, device)

print(f'Best model validation_1 loss: {val_loss_1:.4f}')
print(f'Best model validation_1 accuracy: {val_accuracy_1:.4f}')

In [ ]:
train_loop(model_2, train_dataloader_2, optimizer_2, valid_dataloader_2, 'best_model_2_checkpoint.pth')

model_2.load_state_dict(torch.load('best_model_2_checkpoint.pth'))
model_2.to(device)

# 검증 데이터에 대한 정확도와 손실 계산
val_loss_2, val_accuracy_2 = evaluate(model_2, valid_dataloader_2, criterion, device)

print(f'Best model validation_2 loss: {val_loss_2:.4f}')
print(f'Best model validation_2 accuracy: {val_accuracy_2:.4f}')

In [ ]:
train_loop(model_3, train_dataloader_3, optimizer_3, valid_dataloader_3, 'best_model_3_checkpoint.pth')

model_3.load_state_dict(torch.load('best_model_3_checkpoint.pth'))
model_3.to(device)

# 검증 데이터에 대한 정확도와 손실 계산
val_loss_3, val_accuracy_3 = evaluate(model_3, valid_dataloader_3, criterion, device)

print(f'Best model validation_3 loss: {val_loss_3:.4f}')
print(f'Best model validation_3 accuracy: {val_accuracy_3:.4f}')

In [ ]:
def predict(text, model, word_to_index, index_to_tag):
    # 모델 평가 모드
    model.eval()

    # 토큰화 및 정수 인코딩. OOV 문제 발생 시 <UNK> 토큰에 해당하는 인덱스 1 할당
    tokens = mecab.morphs(text)
    token_indices = [word_to_index.get(token.lower(), 1) for token in tokens]

    # 리스트를 텐서로 변경
    input_tensor = torch.tensor([token_indices], dtype=torch.long).to(device)  # (1, seq_length)

    # 모델의 예측
    with torch.no_grad():
        logits = model(input_tensor)  # (1, output_dim)

    # 레이블 인덱스 예측
    _, predicted_index = torch.max(logits, dim=1)  # (1,)

    # 인덱스와 매칭되는 카테고리 문자열로 변경
    predicted_tag = index_to_tag[predicted_index.item()]

    return predicted_tag


In [ ]:
index_to_tag_0 = {0 : '사실형', 1 : '추론형', 2:'대화형', 3:'예측형'}
index_to_tag_1 = {0 : '긍정', 1 : '부정', 2:'미정'}
index_to_tag_2 = {0 : '과거', 1 : '현재', 2:'미래'}
index_to_tag_3 = {0 : '확실', 1 : '불확실'}

In [ ]:
test_input = "나는 내일 놀러갈거야"
predict_list = [predict(test_input, model_0, word_to_index, index_to_tag_0), predict(test_input, model_1, word_to_index, index_to_tag_1), predict(test_input, model_2, word_to_index, index_to_tag_2), predict(test_input, model_3, word_to_index, index_to_tag_3)]
predict_list

In [ ]:
predict(test_input, model_3, word_to_index, index_to_tag_3)